In [ ]:
import os
import nibabel as nib
import boto3
from botocore import UNSIGNED
from botocore.client import Config
import shutil
from pathlib import Path

from nilearn import datasets

from utils.nsd_utils import NSDDataHandler


BUCKET = "natural-scenes-dataset"
ROOT = "nsddata_timeseries/ppdata"
NIFTI_REL_PATH = 'func1pt8mm/timeseries'

In [2]:
handler = NSDDataHandler()
subj = "subj01"

print("Listing runs...")
runs = handler.list_runs(subj)
print(len(runs))

Listing runs...
548


## ROI Creation

In [13]:
handler.template_dir

'data\\templates'

In [ ]:
from pathlib import Path
from utils.nsd_utils import create_spherical_roi_atlas, VISUAL_CORTEX_ROI_COORDS, VISUAL_CORTEX_ROI_RADIUS

output_dir = Path(handler.template_dir) / "ROIs"
output_path = output_dir / "visual_sphere_atlas_23.nii.gz"
labels_json_path = output_dir / "visual_sphere_atlas_23_labels.json"

label_map = create_spherical_roi_atlas(
    template_path=handler.template_path,
    output_path=output_path,
    roi_coords=VISUAL_CORTEX_ROI_COORDS,
    default_radius=5,
    radius_by_roi=VISUAL_CORTEX_ROI_RADIUS,
    overwrite_overlaps=False,
    labels_json_path=labels_json_path,
)

# Single Subject - Test

In [8]:
subj = 'subj01'

atlas_path = str(Path(handler.template_dir) / "ROIs" / "visual_sphere_atlas_23.nii.gz")
handler = NSDDataHandler(roi_atlas_path = atlas_path)
runs_save_dir = handler.download_runs(subj, 9999, prefix="session")

Starting download of 516 runs for subj01...


100%|██████████| 516/516 [00:00<00:00, 1310.40it/s]


In [9]:
atlas_path

'data/templates/ROIs/visual_sphere_atlas_23.nii.gz'

In [12]:

handler.get_t1_to_epi_warp(subj)
handler.get_warped_atlas(subj, warp_type="func")


Warping ROI atlas to func space ...


'data/nsd_derivatives/subj01/func/atlas_visual_sphere_atlas_23.nii.gz_in_func_subject_space.nii.gz'

In [13]:
saved_paths = handler.extract_all_roi_timeseries(subj, verbose=True)

Found 516 BOLD runs for subj01. Processing ROI timeseries...


Extracting ROI timeseries subj01: 100%|██████████| 516/516 [17:19<00:00,  2.02s/it]

Completed. Processed 516/516 runs successfully.


In [20]:


shutil.rmtree(handler.download_dir)

# Get Subj ROI Timeseries from Atlas

In [18]:
atlas_path = Path(handler.template_dir) / "ROIs" / "visual_sphere_atlas_23.nii.gz"

for i in range(1, 9): # 1
    subj = f"subj0{i}"
    print("--------------------")
    print(f"Started {subj}")
    print("--------------------")
    
    runs_save_dir = handler.download_runs(subj, 9999, prefix="session")
    handler = NSDDataHandler(roi_atlas_path=atlas_path)
    handler.get_t1_to_epi_warp(subj)
    handler.get_warped_atlas(subj, warp_type="func")
    
    saved_paths = handler.extract_all_roi_timeseries(subj, verbose=True)
    
    shutil.rmtree(handler.download_dir)

--------------------
Started subj01
--------------------
Starting download of 516 runs for subj01...


 98%|█████████████████████████████████████████████████████▏| 508/516 [9:39:43<09:07, 68.47s/it]


EndpointConnectionError: Could not connect to the endpoint URL: "https://natural-scenes-dataset.s3.amazonaws.com/nsddata_timeseries/ppdata/subj01/func1pt8mm/timeseries/timeseries_session40_run05.nii.gz"

## Encrpyt Subjects & Sessions

In [ ]:
import importlib, utils.nsd_utils
importlib.reload(utils.nsd_utils)
from utils.nsd_utils import *

In [14]:
handler = NSDDataHandler()
atlas_path = Path(handler.template_dir) / "ROIs" / "visual_sphere_atlas_23.nii.gz"


In [15]:
atlas_name = atlas_path.name.replace(".nii", "")
atlas_name

'visual_sphere_atlas_23.gz'

In [16]:
handler = NSDDataHandler()
handler.train_test_split(atlas_name=atlas_name, test_size=0.2, random_state=46, method="pool", enc_key=b"some_key")

Performing pooled-stratified split...
